# Common · 02 — Reinforcement-learning library

Requires `common/00_config.ipynb`. Action space, reward oracle, state contexts, bandits, DP, Monte Carlo, TD and planning algorithms (24AIM304 Units 1–4).

## `space`

Discrete multi-categorical action space a = (wavelet, level, band, window, fusion).

In [ ]:
import itertools

import numpy as np

COMPONENTS = ["wavelet", "level", "band", "window_s", "fusion"]


class ConfigSpace:
    def __init__(self, cfg: dict):
        A = cfg["actions"]
        self.options = [A["wavelet"], A["level"], [tuple(b) for b in A["band"]], A["window_s"], A["fusion"]]
        self.dims = [len(o) for o in self.options]
        self.n = int(np.prod(self.dims))
        self.radix = np.cumprod([1] + self.dims[::-1][:-1])[::-1]
        b = cfg["baseline_config"]
        self.baseline = self.encode(
            [A["wavelet"].index(b["wavelet"]), A["level"].index(b["level"]), self.options[2].index(tuple(b["band"])),
             A["window_s"].index(b["window_s"]), A["fusion"].index(b["fusion"])]
        )

    def encode(self, idx) -> int:
        return int(np.dot(idx, self.radix))

    def decode(self, c: int) -> list[int]:
        out = []
        for r in self.radix:
            out.append(int(c // r))
            c %= r
        return out

    def values(self, c: int) -> dict:
        return {k: o[i] for k, o, i in zip(COMPONENTS, self.options, self.decode(c))}

    def all_configs(self):
        return range(self.n)

    def feature_keys(self, c: int) -> tuple:
        wi, li, bi, ti, fi = self.decode(c)
        wv, L, _, win, fus = (o[i] for o, i in zip(self.options, (wi, li, bi, ti, fi)))
        keys = []
        if "T" in fus:
            keys.append(("T", win))
        if "S" in fus:
            keys.append(("S", bi, win))
        if "W" in fus:
            keys.append(("W", wv, L, bi, win))
        return tuple(keys)

    def complexity(self, c: int) -> float:
        v = self.values(c)
        branches = len(v["fusion"]) / 3
        depth = v["level"] / max(self.options[1]) if "W" in v["fusion"] else 0.0
        return float(0.5 * branches + 0.3 * depth + 0.2 * v["window_s"] / max(self.options[3]))

    def label(self, c: int) -> str:
        v = self.values(c)
        return f"{v['wavelet']}-L{v['level']}-{v['band'][0]}-{v['band'][1]}Hz-{v['window_s']}s-{v['fusion']}"

    def prefix_sizes(self):
        return [int(np.prod(self.dims[:t])) for t in range(len(self.dims) + 1)]

    def restricted(self, fixed: dict) -> np.ndarray:
        mask = np.ones(self.n, bool)
        for c in range(self.n):
            d = self.decode(c)
            mask[c] = all(d[COMPONENTS.index(k)] == v for k, v in fixed.items())
        return mask


def iter_all(space: ConfigSpace):
    return itertools.product(*[range(d) for d in space.dims])

## `rewards`

Reward oracle: a fast, patient-grouped, out-of-fold proxy classifier for every configuration.

Evaluating a configuration = extract its feature groups -> logistic regression -> probability.
Training-split probabilities are out-of-fold (5-fold, grouped by patient), so the reward an
agent receives for a training recording never comes from a model that saw that recording.

Per-recording reward (Phase 12), with p_i = probability assigned to the true class:
    bal_i  = p_i * 0.5/pi_{y_i}          E[bal]  = balanced accuracy
    sens_i = 1[y_i=1] p_i / pi_1         E[sens] = sensitivity
    spec_i = 1[y_i=0] p_i / pi_0         E[spec] = specificity
    rob_i  = bal_i under 10 dB pink noise
    r_i(c) = .40 bal + .25 sens + .20 spec + .10 rob - .05 complexity(c)
and, with delta_vs_baseline, r_i(c) - r_i(c_baseline)  (the plan's "Delta metric" form).
Macro-F1 is not decomposable per sample, so it is used for policy selection, not as the reward.

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def _clf():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.05, class_weight="balanced", max_iter=3000))


def _fit_one(Xc, Xn, y, groups, tr, va, te, n_folds, seed):
    Xc = np.nan_to_num(Xc, posinf=0, neginf=0)
    Xn = np.nan_to_num(Xn, posinf=0, neginf=0)
    oof_c = np.zeros(len(tr), np.float32)
    oof_n = np.zeros(len(tr), np.float32)
    skf = StratifiedGroupKFold(n_folds, shuffle=True, random_state=seed)
    for a, b in skf.split(tr, y[tr], groups[tr]):
        m = _clf().fit(Xc[tr[a]], y[tr[a]])
        oof_c[b] = m.predict_proba(Xc[tr[b]])[:, 1]
        oof_n[b] = m.predict_proba(Xn[tr[b]])[:, 1]
    m = _clf().fit(Xc[tr], y[tr])
    out = {"train": (oof_c, oof_n)}
    for name, idx in (("val", va), ("test", te)):
        out[name] = (m.predict_proba(Xc[idx])[:, 1].astype(np.float32), m.predict_proba(Xn[idx])[:, 1].astype(np.float32))
    return out


def proxy_probabilities(feats: dict, space, y, groups, tr, va, te, n_folds=5, seed=0, n_jobs=-2):
    ukeys = sorted({space.feature_keys(c) for c in space.all_configs()}, key=str)
    kidx = {k: i for i, k in enumerate(ukeys)}

    def X(kind, key):
        return np.concatenate([feats[kind][g] for g in key], axis=1)

    res = Parallel(n_jobs=n_jobs, verbose=2)(
        delayed(_fit_one)(X("clean", k), X("noisy", k), y, groups, tr, va, te, n_folds, seed) for k in ukeys
    )
    cfg_to_u = np.array([kidx[space.feature_keys(c)] for c in space.all_configs()])
    probs = {}
    for split in ("train", "val", "test"):
        clean = np.stack([r[split][0] for r in res])[cfg_to_u]
        noisy = np.stack([r[split][1] for r in res])[cfg_to_u]
        probs[split] = {"clean": clean, "noisy": noisy}
    return probs


def reward_components(p_clean, p_noisy, y, prior1):
    pc = np.where(y[None] == 1, p_clean, 1 - p_clean)
    pn = np.where(y[None] == 1, p_noisy, 1 - p_noisy)
    w = np.where(y == 1, 0.5 / prior1, 0.5 / (1 - prior1))[None]
    return {
        "balanced": pc * w,
        "sensitivity": np.where(y[None] == 1, pc / prior1, 0.0),
        "specificity": np.where(y[None] == 0, pc / (1 - prior1), 0.0),
        "robustness": pn * w,
    }


def reward_table(comp: dict, complexity: np.ndarray, weights: dict, baseline: int | None) -> np.ndarray:
    R = sum(weights[k] * comp[k] for k in ("balanced", "sensitivity", "specificity", "robustness"))
    R = R - weights["complexity"] * complexity[:, None]
    if baseline is not None:
        R = R - R[baseline][None]
    return np.ascontiguousarray(R.T.astype(np.float32))

## `context`

Discrete RL state: label-free signal descriptors -> PCA (24AIM301 CO3) -> k-means contexts.

The "compact embedding" of the draft plan is realised as the leading PCA components of the
32-band log-PSD rather than a supervised CNN embedding, which would leak label information
into the state and behave differently on training vs. unseen recordings.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler


SCALAR_STATE = ["snr_db", "log_energy", "variance", "centroid", "bandwidth", "zcr", "proc_duration_s",
                "flatness", "spec_entropy", "kurtosis", "ceps_bpm", "ceps_prominence"]


class ContextModel:
    def __init__(self, n_contexts: int, n_pca: int, loc_weight: float = 1.0, seed: int = 0):
        self.K, self.n_pca, self.loc_weight, self.seed = n_contexts, n_pca, loc_weight, seed

    def _raw(self, df: pd.DataFrame) -> np.ndarray:
        psd = df[[c for c in df.columns if c.startswith("logpsd_")]].values
        return np.nan_to_num(np.concatenate([df[SCALAR_STATE].values, psd], axis=1))

    def _loc(self, df):
        return np.stack([(df.location == l).values for l in LOCATIONS], 1).astype(float) * self.loc_weight

    def fit(self, df: pd.DataFrame):
        self.scaler = RobustScaler().fit(self._raw(df))
        self.pca = PCA(self.n_pca, random_state=self.seed).fit(self.scaler.transform(self._raw(df)))
        Z = self.embed(df)
        self.km = KMeans(self.K, n_init=20, random_state=self.seed).fit(Z) if self.K > 1 else None
        return self

    def embed(self, df):
        z = self.pca.transform(self.scaler.transform(self._raw(df)))
        z = z / (np.sqrt(self.pca.explained_variance_[None]) + 1e-8)
        return np.concatenate([z, self._loc(df)], axis=1)

    def predict(self, df) -> np.ndarray:
        if self.K == 1:
            return np.zeros(len(df), int)
        return self.km.predict(self.embed(df))

## `bandits`

Phase 9 â€” wavelet selection as a (contextual) multi-armed bandit (24AIM304 Unit 2).

Xiao & Wang (PLOS ONE 2025) model wavelet-base selection as a *stateless* MDP solved with a
policy gradient; a stateless MDP is exactly a k-armed bandit, and the gradient-bandit
algorithm below is its tabular policy-gradient (REINFORCE-with-baseline) counterpart.
Each pull samples a training recording from the current context and pays the recording's
out-of-fold reward for the chosen arm (configuration).

In [ ]:
import numpy as np


class BanditEnv:
    def __init__(self, R: np.ndarray, ctx: np.ndarray, arms: list[int], K: int, rng):
        self.R, self.ctx, self.arms, self.K, self.rng = R, ctx, np.asarray(arms), K, rng
        self.by_ctx = [np.where(ctx == k)[0] for k in range(K)]
        self.means = np.stack([R[self.by_ctx[k]][:, self.arms].mean(0) for k in range(K)])

    def pull(self, k: int, a: int) -> float:
        i = self.rng.choice(self.by_ctx[k])
        return float(self.R[i, self.arms[a]])

    def sample_context(self) -> int:
        return int(self.ctx[self.rng.integers(len(self.ctx))])


def run_bandit(env: BanditEnv, algo: str, steps: int, rng, **hp):
    K, A = env.K, len(env.arms)
    Q = np.zeros((K, A))
    N = np.zeros((K, A))
    H = np.zeros((K, A))
    rbar = np.zeros(K)
    mu, prec = np.zeros((K, A)), np.full((K, A), 1.0 / hp.get("prior_var", 0.05))
    noise_prec = 1.0 / hp.get("noise_var", 0.05)
    rewards, optimal = np.zeros(steps), np.zeros(steps, bool)
    best = env.means.argmax(1)
    for t in range(steps):
        k = env.sample_context()
        if algo == "eps_greedy":
            a = rng.integers(A) if rng.random() < hp.get("epsilon", 0.1) else int(np.argmax(Q[k]))
        elif algo == "ucb":
            n_k = N[k].sum() + 1
            if (N[k] == 0).any():
                a = int(np.argmin(N[k]))
            else:
                a = int(np.argmax(Q[k] + hp.get("c", 0.1) * np.sqrt(np.log(n_k) / N[k])))
        elif algo == "gradient":
            pi = np.exp(H[k] - H[k].max())
            pi /= pi.sum()
            a = int(rng.choice(A, p=pi))
        elif algo == "thompson":
            a = int(np.argmax(rng.normal(mu[k], 1 / np.sqrt(prec[k]))))
        elif algo == "random":
            a = int(rng.integers(A))
        else:
            raise ValueError(algo)
        r = env.pull(k, a)
        N[k, a] += 1
        Q[k, a] += (r - Q[k, a]) / N[k, a]
        if algo == "gradient":
            rbar_n = N[k].sum()
            rbar[k] += (r - rbar[k]) / rbar_n
            onehot = np.eye(A)[a]
            H[k] += hp.get("alpha", 0.5) * (r - rbar[k]) * (onehot - pi)
        if algo == "thompson":
            new_prec = prec[k, a] + noise_prec
            mu[k, a] = (prec[k, a] * mu[k, a] + noise_prec * r) / new_prec
            prec[k, a] = new_prec
        rewards[t] = r
        optimal[t] = a == best[k]
    score = H if algo == "gradient" else (mu if algo == "thompson" else Q)
    policy = score.argmax(1)
    return {"policy_arm": policy, "policy_config": env.arms[policy], "rewards": rewards, "optimal": optimal, "N": N}

## `tabular`

Phase 10/12 â€” representation selection as a finite-horizon MDP, solved with tabular RL.

MDP (24AIM304 Unit 2):
  s_0 = (context k, nothing chosen)                  k = discrete signal context of the recording
  s_t = (k, a_0..a_{t-1})                            a prefix of the configuration
  actions at step t: options of component t          wavelet -> level -> band -> window -> fusion
  transitions: deterministic prefix extension; the recording (hence reward) is the stochastic part
  reward: 0 for t < 4, r_i(config) at the terminal step; gamma = 1, horizon H = 5

Solvers
  DP (Unit 2):          value iteration, policy iteration on the empirical model  (planning optimum)
  Monte Carlo (Unit 3): on-policy first-visit MC control with epsilon-soft policies
  TD (Unit 3):          SARSA, Expected SARSA, Q-learning, Double Q-learning, n-step SARSA
  Planning (Unit 4):    Dyna-Q, prioritized sweeping, rollout algorithm, MCTS/UCT (heuristic search)
Every learner only ever sees sampled (recording, reward) pairs; the empirical model is used by
DP / as the ground truth for learning curves.

In [ ]:
import heapq
import math

import numpy as np

H = 5


class ConfigMDP:
    def __init__(self, R: np.ndarray, ctx: np.ndarray, K: int, dims: list[int], allowed=None, rng=None):
        self.R, self.ctx, self.K, self.dims = R, ctx, K, list(dims)
        self.P = [int(np.prod(dims[:t])) for t in range(H + 1)]
        self.allowed = [np.arange(d) if allowed is None or allowed[t] is None else np.asarray(allowed[t])
                        for t, d in enumerate(dims)]
        self.amask = [np.isin(np.arange(d), al) for d, al in zip(dims, self.allowed)]
        self.rng = rng or np.random.default_rng(0)
        self.by_ctx = [np.where(ctx == k)[0] for k in range(K)]
        self.Rbar = np.stack([R[ix].mean(0) if len(ix) else np.zeros(R.shape[1]) for ix in self.by_ctx])
        self.leaf_mask = self._leaf_mask()

    def _leaf_mask(self):
        m = np.ones(self.dims, bool)
        for t, am in enumerate(self.amask):
            shape = [1] * H
            shape[t] = self.dims[t]
            m &= am.reshape(shape)
        return m.reshape(-1)

    def sample(self):
        i = int(self.rng.integers(len(self.ctx)))
        return i, int(self.ctx[i])

    def sample_in_ctx(self, k):
        return int(self.rng.choice(self.by_ctx[k]))

    def new_q(self, init=0.0):
        return [np.full((self.K, self.P[t], self.dims[t]), init, np.float64) for t in range(H)]

    def greedy(self, q_row, t):
        v = np.where(self.amask[t], q_row, -np.inf)
        return int(np.argmax(v))

    def eps_greedy(self, q_row, t, eps):
        if self.rng.random() < eps:
            return int(self.rng.choice(self.allowed[t]))
        return self.greedy(q_row, t)

    def policy_from_q(self, Q) -> np.ndarray:
        out = np.zeros(self.K, int)
        for k in range(self.K):
            p = 0
            for t in range(H):
                p = p * self.dims[t] + self.greedy(Q[t][k, p], t)
            out[k] = p
        return out

    def policy_value(self, policy: np.ndarray) -> float:
        w = np.array([len(ix) for ix in self.by_ctx]) / len(self.ctx)
        return float(sum(w[k] * self.Rbar[k, policy[k]] for k in range(self.K)))


def value_iteration(mdp: ConfigMDP, theta=1e-10, max_sweeps=100):
    V = [np.zeros((mdp.K, mdp.P[t])) for t in range(H + 1)]
    leaf = np.where(mdp.leaf_mask[None], mdp.Rbar, -np.inf)
    history = []
    V[H] = leaf
    for sweep in range(max_sweeps):
        newV = list(V)
        delta = 0.0
        for t in range(H):
            child = np.where(mdp.amask[t][None, None], V[t + 1].reshape(mdp.K, mdp.P[t], mdp.dims[t]), -np.inf)
            newV[t] = child.max(-1)
            fin = np.isfinite(newV[t])
            delta = max(delta, float(np.abs(newV[t][fin] - V[t][fin]).max(initial=0)))
        V = newV
        history.append(delta)
        if delta < theta:
            break
    policy = leaf.argmax(1)
    return {"policy": policy, "V0": V[0][:, 0], "sweeps": len(history), "deltas": history}


def policy_iteration(mdp: ConfigMDP, rng=None, max_iter=50):
    rng = rng or np.random.default_rng(0)
    leaf = np.where(mdp.leaf_mask[None], mdp.Rbar, -np.inf)
    pol = [np.stack([rng.choice(mdp.allowed[t], mdp.P[t]) for _ in range(mdp.K)]) for t in range(H)]
    for it in range(max_iter):
        Qs = [None] * H
        Vn = leaf
        for t in reversed(range(H)):
            Qs[t] = np.where(mdp.amask[t][None, None], Vn.reshape(mdp.K, mdp.P[t], mdp.dims[t]), -np.inf)
            Vn = np.take_along_axis(Qs[t], pol[t][..., None], -1)[..., 0]
        new = [q.argmax(-1) for q in Qs]
        stable = all((a == b).all() for a, b in zip(new, pol))
        pol = new
        if stable:
            break
    Q = [np.where(np.isfinite(q), q, -1e9) for q in Qs]
    return {"policy": mdp.policy_from_q(Q), "iterations": it + 1}


def _alpha(n, alpha):
    return max(alpha, 1.0 / n)


def _eps(ep, episodes, sched):
    frac = min(1.0, ep / (sched["decay_frac"] * episodes))
    return sched["start"] + frac * (sched["end"] - sched["start"])


def run_tabular(mdp: ConfigMDP, method: str, episodes: int, alpha=0.1, eps_sched=None, n=3,
                planning=10, eval_every=1000, val_fn=None, theta=1e-4):
    eps_sched = eps_sched or {"start": 1.0, "end": 0.05, "decay_frac": 0.7}
    Q = mdp.new_q()
    Q2 = mdp.new_q() if method == "double_q" else None
    N = mdp.new_q()
    model_r = np.zeros((mdp.K, mdp.P[H]))
    model_n = np.zeros((mdp.K, mdp.P[H]))
    seen, seen_set, pq = [], set(), []
    curve = []

    def qrow(t, k, p):
        return Q[t][k, p] + (Q2[t][k, p] if Q2 is not None else 0)

    def pi_probs(t, k, p, eps):
        al = mdp.allowed[t]
        pr = np.zeros(mdp.dims[t])
        pr[al] = eps / len(al)
        pr[mdp.greedy(qrow(t, k, p), t)] += 1 - eps
        return pr

    def backup_target(t, k, p2, eps, a2=None, table=None):
        T = Q if table is None else table
        if method in ("q_learning", "dyna_q", "prioritized_sweeping"):
            return float(np.max(np.where(mdp.amask[t + 1], T[t + 1][k, p2], -np.inf)))
        if method == "expected_sarsa":
            return float(pi_probs(t + 1, k, p2, eps) @ Q[t + 1][k, p2])
        return float(T[t + 1][k, p2, a2])

    def model_update(t, k, p, a):
        p2 = p * mdp.dims[t] + a
        if t == H - 1:
            target = model_r[k, p2]
        else:
            target = float(np.max(np.where(mdp.amask[t + 1], Q[t + 1][k, p2], -np.inf)))
        err = target - Q[t][k, p, a]
        Q[t][k, p, a] += alpha * err
        return err

    for ep in range(episodes):
        eps = _eps(ep, episodes, eps_sched)
        i, k = mdp.sample()
        p = 0
        traj = []
        a = mdp.eps_greedy(qrow(0, k, 0), 0, eps)
        for t in range(H):
            p2 = p * mdp.dims[t] + a
            a2 = mdp.eps_greedy(qrow(t + 1, k, p2), t + 1, eps) if t < H - 1 else None
            traj.append((t, p, a))
            if t == H - 1:
                r = float(mdp.R[i, p2])
                model_n[k, p2] += 1
                model_r[k, p2] += (r - model_r[k, p2]) / model_n[k, p2]
            if method in ("sarsa", "expected_sarsa", "q_learning", "dyna_q", "prioritized_sweeping"):
                N[t][k, p, a] += 1
                target = r if t == H - 1 else backup_target(t, k, p2, eps, a2)
                err = target - Q[t][k, p, a]
                if method == "prioritized_sweeping":
                    if abs(err) > theta:
                        heapq.heappush(pq, (-abs(err), t, k, p, a))
                else:
                    Q[t][k, p, a] += _alpha(N[t][k, p, a], alpha) * err
                if method == "dyna_q" and (t, k, p, a) not in seen_set:
                    seen_set.add((t, k, p, a))
                    seen.append((t, k, p, a))
            elif method == "double_q":
                N[t][k, p, a] += 1
                A_, B_ = (Q, Q2) if mdp.rng.random() < 0.5 else (Q2, Q)
                if t == H - 1:
                    target = r
                else:
                    a_star = mdp.greedy(A_[t + 1][k, p2], t + 1)
                    target = float(B_[t + 1][k, p2, a_star])
                A_[t][k, p, a] += _alpha(N[t][k, p, a], alpha) * (target - A_[t][k, p, a])
            p, a = p2, a2

        if method == "mc":
            for (t, pp, aa) in traj:
                N[t][k, pp, aa] += 1
                Q[t][k, pp, aa] += (r - Q[t][k, pp, aa]) / N[t][k, pp, aa]
        elif method == "nstep_sarsa":
            pref = [pp for (_, pp, _) in traj]
            acts = [aa for (_, _, aa) in traj]
            for tau in range(H):
                if tau + n >= H:
                    G = r
                else:
                    G = Q[tau + n][k, pref[tau + n], acts[tau + n]]
                N[tau][k, pref[tau], acts[tau]] += 1
                Q[tau][k, pref[tau], acts[tau]] += _alpha(N[tau][k, pref[tau], acts[tau]], alpha) * (G - Q[tau][k, pref[tau], acts[tau]])
        elif method == "dyna_q":
            for _ in range(planning):
                model_update(*seen[mdp.rng.integers(len(seen))])
        elif method == "prioritized_sweeping":
            for _ in range(planning):
                if not pq:
                    break
                _, t, kk, pp, aa = heapq.heappop(pq)
                model_update(t, kk, pp, aa)
                if t > 0:
                    par, pa = divmod(pp, mdp.dims[t - 1])
                    tgt = float(np.max(np.where(mdp.amask[t], Q[t][kk, pp], -np.inf)))
                    pr = abs(tgt - Q[t - 1][kk, par, pa])
                    if pr > theta:
                        heapq.heappush(pq, (-pr, t - 1, kk, par, pa))

        if eval_every and (ep + 1) % eval_every == 0:
            QQ = [Q[t] + Q2[t] for t in range(H)] if Q2 is not None else Q
            pol = mdp.policy_from_q(QQ)
            row = {"episode": ep + 1, "train_value": mdp.policy_value(pol)}
            if val_fn is not None:
                row.update(val_fn(pol))
            curve.append(row)

    QQ = [Q[t] + Q2[t] for t in range(H)] if Q2 is not None else Q
    return {"Q": QQ, "policy": mdp.policy_from_q(QQ), "curve": curve}


def rollout_policy(mdp: ConfigMDP, sims_per_action=20):
    policy = np.zeros(mdp.K, int)
    for k in range(mdp.K):
        p = 0
        for t in range(H):
            best, best_a = -np.inf, None
            for a in mdp.allowed[t]:
                tot = 0.0
                for _ in range(sims_per_action):
                    q = p * mdp.dims[t] + a
                    for u in range(t + 1, H):
                        q = q * mdp.dims[u] + int(mdp.rng.choice(mdp.allowed[u]))
                    tot += mdp.R[mdp.sample_in_ctx(k), q]
                if tot / sims_per_action > best:
                    best, best_a = tot / sims_per_action, a
            p = p * mdp.dims[t] + best_a
        policy[k] = p
    return policy


def mcts_policy(mdp: ConfigMDP, simulations_per_ctx=5000, c=0.05):
    policy = np.zeros(mdp.K, int)
    for k in range(mdp.K):
        Nn, W = {}, {}
        for _ in range(simulations_per_ctx):
            p, path = 0, []
            for t in range(H):
                al = mdp.allowed[t]
                n_s = sum(Nn.get((t, p, a), 0) for a in al) + 1
                scores = []
                for a in al:
                    na = Nn.get((t, p, a), 0)
                    scores.append(np.inf if na == 0 else W[(t, p, a)] / na + c * math.sqrt(math.log(n_s) / na))
                a = int(al[int(np.argmax(scores))])
                path.append((t, p, a))
                p = p * mdp.dims[t] + a
            r = mdp.R[mdp.sample_in_ctx(k), p]
            for key in path:
                Nn[key] = Nn.get(key, 0) + 1
                W[key] = W.get(key, 0.0) + r
        p = 0
        for t in range(H):
            al = mdp.allowed[t]
            a = int(al[int(np.argmax([Nn.get((t, p, a), 0) for a in al]))])
            p = p * mdp.dims[t] + a
        policy[k] = p
    return policy

## `problem`

The assembled RL problem: labelled records, proxy probabilities, contexts, reward tables.

In [ ]:
import numpy as np
import pandas as pd


class RLProblem:
    def __init__(self, cfg, P, K=None, weights=None, delta=None, contexts_file="contexts.csv", oracle=None):
        self.cfg, self.space = cfg, ConfigSpace(cfg)
        self.oracle = oracle or cfg["reward"]["oracle"]
        pr = np.load(P["processed"] / {"deep": "deep_probs.npz", "classical": "proxy_probs.npz"}[self.oracle])
        self.meta = pd.read_csv(P["processed"] / "proxy_records.csv")
        ctx = pd.read_csv(P["processed"] / contexts_file)
        self.meta = self.meta.merge(ctx, on="record_id", how="left")
        self.K = K or cfg["context"]["n_contexts"]
        self.idx = {s: np.where(self.meta.split == s)[0] for s in ("train", "val", "test")}
        self.y = {s: self.meta.y.values[self.idx[s]].astype(int) for s in self.idx}
        self.ctx = {s: self.meta[f"ctx_K{self.K}"].values[self.idx[s]].astype(int) for s in self.idx}
        self.p = {s: {k: pr[f"{s}_{k}"] for k in ("clean", "noisy")} for s in self.idx}
        self.prior1 = self.y["train"].mean()
        self.complexity = np.array([self.space.complexity(c) for c in self.space.all_configs()])
        self.set_reward(weights, delta)

    def set_reward(self, weights=None, delta=None):
        self.weights = weights or self.cfg["reward"]["weights"]
        self.delta = self.cfg["reward"]["delta_vs_baseline"] if delta is None else delta
        base = self.space.baseline if self.delta else None
        self.R = {}
        for s in self.idx:
            comp = reward_components(self.p[s]["clean"], self.p[s]["noisy"], self.y[s], self.prior1)
            self.R[s] = reward_table(comp, self.complexity, self.weights, base)
        return self

    def mdp(self, split="train", allowed=None, seed=0):
        return ConfigMDP(self.R[split], self.ctx[split], self.K, self.space.dims, allowed, np.random.default_rng(seed))

    def evaluate(self, policy, split="val", noisy=False, thr=0.5):
        c = np.asarray(policy)[self.ctx[split]]
        p = self.p[split]["noisy" if noisy else "clean"][c, np.arange(len(c))]
        m = binary_metrics(self.y[split], p, thr)
        m["mean_reward"] = float(self.R[split][np.arange(len(c)), c].mean())
        return m

    def evaluate_config(self, c, split="val", noisy=False):
        return self.evaluate(np.full(self.K, c), split, noisy)

    def allowed_fixed(self, **fixed):
        base = self.space.decode(self.space.baseline)
        allowed = [None] * 5
        for k in fixed:
            j = COMPONENTS.index(k)
            v = fixed[k]
            allowed[j] = [base[j]] if v == "baseline" else list(v)
        return allowed

    def describe_policy(self, policy):
        rows = []
        for k, c in enumerate(policy):
            v = self.space.values(int(c))
            n_tr = int((self.ctx["train"] == k).sum())
            rows.append({"context": k, "n_train": n_tr, "config": self.space.label(int(c)), **{kk: str(vv) for kk, vv in v.items()}})
        return pd.DataFrame(rows)

In [ ]:
print("loaded 02_rl_lib")